# Power BI Copilot Diagnostics — Parser & Troubleshooter

This notebook parses a **Power BI Copilot diagnostic** file (`copilot_diagnostic_data_*.txt`, a JSON document) and turns it into an actionable troubleshooting report. It highlights the signals that tell you **what to adjust in your semantic model** so Copilot answers correctly.

It surfaces:

1. **Session metadata** — who/what/when, agent, service version.
2. **Conversation turns** — each user question, the tool Copilot chose, and whether it succeeded.
3. **Per-question diagnostics** — warnings (e.g. `DataIndexNotReady`, `QueryAggregateNotSupported`, `AgentSchemaReduced`), NL‑to‑DAX **fallback reasons**, generated **DAX**, empty results, and clarifications.
4. **Model AI settings** — the AI data schema (visible vs. hidden fields), AI instructions, example prompts, and indexing state.
5. **An automated recommendations report** mapping each detected issue to the fix, following Microsoft's *Prep data for AI* guidance.
6. **A shareable Markdown report** you can hand to model owners.

**Reference docs**
- [Copilot for Power BI overview](https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-introduction) — the full list of error / warning / info messages.
- [FAQ: Preparing Data for AI (Power BI)](https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-prepare-data-ai-faq)
- [Tutorial: Prepare Semantic Model for AI](https://learn.microsoft.com/en-us/power-bi/create-reports/tutorial-copilot-power-bi-prepare-model)

> Tip: In Copilot, download the diagnostics from the response's *…* menu. Warnings like `AgentSchemaReduced` only appear in these files.


## 1. Load the diagnostic file

By default this finds the most recent `copilot_diagnostic_data_*.txt` next to the notebook. Set `DIAG_PATH` explicitly to point at a specific file.

In [ ]:
from datetime import datetime
import glob
import os
import json

# DIAG_PATH may be: a specific .txt file, a FOLDER (newest matching file is picked),
# or None (search this notebook's folder). Use a raw string (r"...") for Windows paths.
DIAG_PATH = None
# Example: DIAG_PATH = r"C:\path\to\copilot_diagnostic_data_...txt"

PATTERN = 'copilot_diagnostic_data_*.txt'

def _newest_in(folder):
    matches = sorted(
        glob.glob(os.path.join(folder, PATTERN)),
        key=os.path.getmtime,
        reverse=True,
    )
    return matches[0] if matches else None

if DIAG_PATH is None:
    DIAG_PATH = _newest_in(os.getcwd())
elif os.path.isdir(DIAG_PATH):
    # A folder was given — pick the most recent diagnostic file inside it.
    DIAG_PATH = _newest_in(DIAG_PATH)

if not DIAG_PATH or not os.path.isfile(DIAG_PATH):
    raise FileNotFoundError(
        f'No {PATTERN} file found. Set DIAG_PATH to a specific diagnostic .txt file '
        'or to the folder that contains one.'
    )

with open(DIAG_PATH, 'r', encoding='utf-8') as f:
    diag = json.load(f)

print(f'Loaded: {DIAG_PATH}')
print(f'Top-level keys: {list(diag.keys())}')

## 2. Session metadata

In [ ]:
meta = {
    'Session ID': diag.get('sessionId'),
    'Copilot session ID': diag.get('copilotSessionId'),
    'Conversation ID': diag.get('conversationId'),
    'Consumption method': diag.get('consumptionMethod'),
    'Copilot agent': diag.get('CopilotAgentId'),
    'Service version': diag.get('serviceVersion'),
    'Timestamp': diag.get('timestamp'),
}

width = max(len(k) for k in meta)
for k, v in meta.items():
    print(f'{k:<{width}} : {v}')

## 3. Conversation turns

Walk `chatHistory` and pair each **user** question with the **tool** Copilot invoked and the outcome. `answerDataQuestion` = a data/Q&A question; `getDatasetSchema` = a schema/authoring request.

In [ ]:
def text_of(content):
    """Flatten the various content shapes used in chatHistory to plain text."""
    if isinstance(content, str):
        return content
    parts = []
    if isinstance(content, list):
        for item in content:
            if not isinstance(item, dict):
                continue
            if item.get('type') == 'text':
                parts.append(item.get('text', {}).get('value', ''))
    return ' '.join(p for p in parts if p).strip()

turns = []
history = diag.get('chatHistory', [])
for i, msg in enumerate(history):
    role = msg.get('role')
    if role == 'user':
        turns.append({
            'turn': len(turns) + 1,
            'question': text_of(msg.get('content')),
            'tools': [],
            'status': None,
            'answer': None,
        })
    elif role == 'assistant' and msg.get('tool_calls') and turns:
        for tc in msg['tool_calls']:
            turns[-1]['tools'].append(tc.get('function', {}).get('name'))
    elif role == 'tool' and turns:
        result = msg.get('result', {})
        turns[-1]['status'] = result.get('status', {}).get('kind')
        ans = text_of(msg.get('content'))
        if ans:
            turns[-1]['answer'] = ans

for t in turns:
    tools = ', '.join(x for x in t['tools'] if x) or '—'
    print(f"Turn {t['turn']}: {t['question']}")
    print(f"   tool(s): {tools}   status: {t['status']}")
    if t['answer']:
        preview = t['answer'][:220] + ('…' if len(t['answer']) > 220 else '')
        print(f"   answer : {preview}")
    print()

## 4. Per-question diagnostics

The `dataQuestion` object holds the deep signals for each Q&A attempt: interpretation **warnings**, whether Copilot fell back to **NL‑to‑DAX** (and why), the generated **DAX**, whether the result was **empty**, and any **clarification** it returned to the user.

In [ ]:
def deep_get(d, *keys, default=None):
    for k in keys:
        if isinstance(d, dict):
            d = d.get(k)
        elif isinstance(d, list) and isinstance(k, int) and -len(d) <= k < len(d):
            d = d[k]
        else:
            return default
    return d if d is not None else default

# Clarifications for a turn are recorded in *later* turns' contextEvents (as history),
# keyed by the utterance they belong to. Build a global map utterance -> clarificationKind.
clarification_by_utterance = {}
for q in (diag.get('dataQuestion') or {}).values():
    for ev in deep_get(q, 'interpretRequest', 'conversationalContext', 'contextEvents', default=[]):
        ck = deep_get(ev, 'responses', 0, 'command', 'clarification', 'clarificationKind')
        if ck and ev.get('utterance'):
            clarification_by_utterance[ev['utterance']] = ck

questions = []
for qid, q in (diag.get('dataQuestion') or {}).items():
    resp = q.get('interpretResponse', {})
    nl = q.get('nlToDaxDetails', {})
    activity = nl.get('activityDetails', {})

    clar_kinds = []
    ck = clarification_by_utterance.get(q.get('utterance'))
    if ck:
        clar_kinds.append(ck)

    dax_list = nl.get('daxGeneration', []) or []
    dax_query = next((d.get('daxQuery') for d in dax_list if isinstance(d, dict) and d.get('daxQuery')), None)

    questions.append({
        'id': qid,
        'utterance': q.get('utterance'),
        'warnings': resp.get('warnings', []) or [],
        'errors': resp.get('errors', []) or [],
        'restatements': [r for r in (resp.get('restatements') or []) if r],
        'fallback_reason': activity.get('fallbackReason'),
        'empty_result': activity.get('emptyResult'),
        'truncated_result': activity.get('truncatedResult'),
        'dax': dax_query,
        'answer': deep_get(nl, 'textualAnswer', 'answer'),
        'context_clarifications': clar_kinds,
    })

for q in questions:
    print('=' * 90)
    print(f"Q: {q['utterance']}")
    print(f"   warnings         : {q['warnings'] or '—'}")
    print(f"   errors           : {q['errors'] or '—'}")
    if q['restatements']:
        print(f"   restated as      : {q['restatements']}")
    if q['fallback_reason']:
        print(f"   NL->DAX fallback : {q['fallback_reason']}")
    if q['empty_result']:
        print(f"   empty result     : {q['empty_result']}")
    if q['context_clarifications']:
        print(f"   clarification    : {q['context_clarifications']}")
    if q['answer']:
        print(f"   answer           : {q['answer']}")
    if q['dax']:
        print('   generated DAX    :')
        for line in q['dax'].splitlines():
            print(f'       {line}')
    print()

## 5. Troubleshooting knowledge base

**Purpose of this tool:** help you understand **why some prompts are answered well while others aren't**, so you can remediate the **source — the semantic model** — for consistent, high-quality Copilot responses.

Each known signal (warning, **info message**, fallback reason, or clarification) maps to what it means and **what to adjust in the model**, per the *Prep data for AI* guidance.

**Exact codes often have no dedicated page** on Microsoft Learn — that's expected. Rather than looking up each string verbatim, the tool reads the code and maps it to the closest **remediation category** by keyword (e.g. *any* schema-related signal → **simplify the semantic model schema**; any index-related signal → **enable/complete Q&A indexing**). Known codes come from `KB`; unknown ones fall through to `infer_guidance()` / `GUIDANCE_HEURISTICS`.

Some signals in the diagnostic's `warnings` array are **informational only** (e.g. `VerifiedAnswerNaiveMatch`) — these are listed in `INFO_SIGNALS` and reported at **INFO** severity, not WARNING. For the full list of Copilot error / warning / info messages, see the [Copilot for Power BI overview](https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-introduction).


In [ ]:
FAQ = 'https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-prepare-data-ai-faq'
TUT = 'https://learn.microsoft.com/en-us/power-bi/create-reports/tutorial-copilot-power-bi-prepare-model'
# General Copilot for Power BI overview — reference for all error / warning / info messages.
OVERVIEW = 'https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-introduction'

# Signals that are informational only (surfaced in the diagnostic's `warnings` array but not an
# actual problem). They are reported at INFO severity, not WARNING, in section 7.
INFO_SIGNALS = {'VerifiedAnswerNaiveMatch'}

KB = {
    # --- Interpretation warnings ---
    'DataIndexNotReady': {
        'meaning': "The semantic model's AI index isn't ready (model is indexing/syncing). Copilot may miss instance values (e.g. specific product or region names) until indexing completes.",
        'fix': "Wait for indexing to finish, then retry. Ensure Q&A is enabled (IndexingEnabled) so instance values are indexed. Republish or refresh to trigger reindexing. The first index can take ~15 extra minutes; import models reindex on refresh, DirectQuery/Direct Lake at most once per 24h.",
        'link': FAQ,
    },
    'QueryAggregateNotSupported': {
        'meaning': 'Q&A could not perform the requested aggregation/calculation over the fields as asked, so it could not answer natively (often triggering an NL-to-DAX fallback or a partial answer).',
        'fix': 'Add an explicit measure for the calculation the user wants, then add an AI instruction mapping the phrasing to that measure. For common asks, set a Verified Answer on a visual that already computes it.',
        'link': FAQ,
    },
    'AgentSchemaReduced': {
        'meaning': 'The model schema was too large for Copilot to process, so it automatically reduced the schema. Fields you expect may be dropped from consideration.',
        'fix': 'Simplify the AI data schema: hide tables/columns/measures that end users never query. Keep within limits (~1,000 model entities and 5M instance values). Remove duplicate/similar field names across tables.',
        'link': FAQ,
    },
    'AgentSchemaHeavilyReduced': {
        'meaning': 'The model schema was far too large, so Copilot received a heavily reduced view of it. Many fields/measures the question needs may be missing, which hurts accuracy and can cause failures on complex asks.',
        'fix': 'Aggressively simplify the AI data schema: hide tables/columns/measures users never query, consolidate per-level/per-version measures (calculation groups), remove duplicate/similar names, and stay within limits (~1,000 entities, 5M instance values).',
        'link': FAQ,
    },
    'LinguisticSchemaTruncated': {
        'meaning': 'The linguistic schema sent to the AI was cut short because the model is too large. Copilot cannot see the full model and may miss the right field/measure for a question.',
        'fix': 'Reduce the model size exposed to AI — hide unused tables/columns/measures and consolidate the measure architecture — so the linguistic schema fits without truncation.',
        'link': FAQ,
    },
    'DataIndexLimitReached': {
        'meaning': 'Indexing hit the upper bound (5M instance values). Copilot answers from a partial index, so some instance values are not searchable.',
        'fix': 'Hide high-cardinality text columns that are not needed for questions from the AI schema, and avoid indexing very large lookup columns. Split or reduce the model so key columns index first (smallest cardinality is indexed first).',
        'link': FAQ,
    },
    'QueryNotSupported': {
        'meaning': 'Copilot could not translate the question into a supported query/DAX plan. Usually the ask is highly multi-dimensional (multiple versions x revenue types x time grains x change formats at once) and the reduced/truncated schema did not give it enough to plan over.',
        'fix': 'Reduce what the AI must reason over: simplify/prune the AI schema so it is not truncated; add explicit measures for the requested calculations; pin Verified Answers for the common multi-dimensional asks; and add AI instructions mapping the phrasing to those measures. Users can also split one complex prompt into a few simpler ones.',
        'link': FAQ,
    },
    # --- Informational messages (not problems) ---
    'VerifiedAnswerNaiveMatch': {
        'meaning': 'Informational, not a warning. Copilot matched the question to a Verified Answer using a simple (naive) text match on the trigger phrase rather than a semantic match. The verified answer was still served, but the loose match means slightly different phrasings might not trigger it.',
        'fix': 'No action required if the served answer is correct. To make matching more robust, refine the Verified Answer trigger phrases to cover the phrasings users actually type, or add AI instructions/synonyms so related wording maps to the same verified answer.',
        'link': OVERVIEW,
    },
    # --- NL-to-DAX fallback reasons ---
    'emptyInterpretation': {
        'meaning': 'Q&A could not interpret the question against the model, so Copilot fell back to generating DAX directly. This usually means naming/terminology or schema gaps.',
        'fix': 'Improve field/table names (human-readable, unique), add descriptions, and add synonyms/Terms for the words users actually use. Add AI instructions for business terms and define verified answers for common questions.',
        'link': TUT,
    },
    'queryNotSupported': {
        'meaning': 'Q&A could not natively answer the question (too complex / not supported), so Copilot fell back to generating DAX directly. The answer may be less reliable.',
        'fix': 'Simplify the AI schema so it is not truncated, add explicit measures for the requested calculations, and pin Verified Answers for common multi-dimensional asks. Users can also break one complex prompt into simpler ones.',
        'link': FAQ,
    },
    # --- Clarification kinds ---
    'AdvancedQueryLimitation': {
        'meaning': 'The user asked for something the model cannot do (e.g. forecasting/prediction of future values, or an advanced calculation not present in the model).',
        'fix': 'If the capability is expected, add it to the model (e.g. a forecast measure or calculation). Otherwise set user expectations via AI instructions describing what the model can/cannot answer.',
        'link': FAQ,
    },
}

# Signals that are not a named warning but still worth flagging.
EMPTY_RESULT_GUIDANCE = {
    'meaning': 'The generated DAX ran but returned a blank/zero result. Either the data genuinely has no rows for that filter, or a field/term was mapped incorrectly (wrong table, relationship, or value).',
    'fix': 'Confirm the data exists for that filter. If mapping is wrong, add AI instructions to map the term to the correct field/measure, verify relationships (star schema), and ensure the value is indexed (Q&A enabled).',
    'link': TUT,
}

# --- Heuristic category matching -------------------------------------------------------
# Copilot emits many warning / info codes and the exact string often has NO dedicated page
# on Microsoft Learn. That's expected. The point of this tool is to help you understand *why*
# some prompts are answered well and others aren't, so you can remediate the SOURCE (the
# semantic model) for consistent, high-quality Copilot answers. So when a code isn't in KB
# above, classify it by keyword into the closest remediation category (e.g. anything mentioning
# "schema" -> simplify the model schema) rather than giving up. Ordered most-specific first;
# the first matching token group wins.
GUIDANCE_HEURISTICS = [
    (('schema', 'entity', 'entities', 'toolarge', 'toomany', 'reduced', 'truncat', 'linguistic'), {
        'meaning': 'Schema-scale signal: the AI data schema is likely too large or ambiguous, so Copilot cannot reliably reason over all of it and may drop or confuse fields.',
        'fix': 'Simplify the semantic model schema exposed to AI — hide tables/columns/measures users never query, remove duplicate or similar field names, and stay within limits (~1,000 entities, 5M instance values).',
        'link': FAQ,
    }),
    (('index', 'indexing', 'notready', 'sync', 'qna', 'q&a'), {
        'meaning': 'Indexing signal: the model AI index is not ready or incomplete, so Copilot cannot search instance values (specific names/values) reliably.',
        'fix': 'Enable Q&A / indexing on the model and allow the first index to finish (~15 min). Reindex by refreshing/republishing. Reduce high-cardinality text columns so key columns index first.',
        'link': FAQ,
    }),
    (('aggregate', 'measure', 'calc', 'compute', 'sum', 'average'), {
        'meaning': 'Calculation signal: Copilot could not compute the requested aggregation/metric from the exposed fields.',
        'fix': 'Add an explicit measure for the calculation, then add an AI instruction mapping the user phrasing to that measure. For common asks, pin a Verified Answer on a visual that already computes it.',
        'link': FAQ,
    }),
    (('verified', 'verifiedanswer'), {
        'meaning': 'Verified Answer signal: Copilot involved (or loosely matched) a Verified Answer for this question.',
        'fix': 'If the answer is correct, no action needed. To make matching robust, refine Verified Answer trigger phrases and add synonyms/AI instructions so related wording maps to the same verified answer.',
        'link': OVERVIEW,
    }),
    (('notsupported', 'unsupported', 'querynotsupported', 'complex', 'toocomplex'), {
        'meaning': 'Unsupported/too-complex query signal: Copilot could not plan a query for this ask, typically because it is highly multi-dimensional and the schema it saw was reduced/truncated.',
        'fix': 'Simplify/prune the AI schema so it is not truncated, add explicit measures for the requested calculations, and pin Verified Answers for common multi-dimensional asks. Users can also break one complex prompt into simpler ones.',
        'link': FAQ,
    }),
    (('interpret', 'nltodax', 'nl2dax', 'dax', 'fallback', 'understand'), {
        'meaning': 'Interpretation signal: Q&A could not confidently interpret the question against the model, so Copilot fell back to raw DAX generation (less reliable).',
        'fix': 'Improve field/table naming (human-readable, unique), add descriptions and synonyms/Terms for the words users actually use, and add AI instructions for business terms.',
        'link': TUT,
    }),
    (('term', 'synonym', 'naming', 'name', 'ambiguous', 'ambiguity'), {
        'meaning': 'Naming/terminology signal: the vocabulary in the question does not clearly map to model field names, or names are ambiguous.',
        'fix': 'Rename fields to be human-readable and unique, add synonyms/Terms for user vocabulary, and add AI instructions mapping business terms to the right field/measure.',
        'link': TUT,
    }),
    (('clarif',), {
        'meaning': 'Clarification signal: Copilot had to ask the user to clarify because the request was ambiguous or beyond the model.',
        'fix': 'Reduce ambiguity in the model (unique names, clear relationships), and use AI instructions/example prompts to steer common phrasings to the right interpretation.',
        'link': FAQ,
    }),
    (('empty', 'noresult', 'blank', 'norow', 'zero'), EMPTY_RESULT_GUIDANCE),
    (('relationship', 'join', 'cardinality', 'star'), {
        'meaning': 'Relationship/modeling signal: the model relationships may be missing, ambiguous, or not a clean star schema, producing wrong or blank results.',
        'fix': 'Follow a star schema, fix or remove ambiguous relationships, and verify the join path used by the question so filters propagate correctly.',
        'link': TUT,
    }),
]


def infer_guidance(signal, is_info=False):
    """Best-effort category guidance for a code not explicitly in KB.

    Exact codes rarely have a dedicated Learn page — match on keywords to the closest
    remediation category so you still get an actionable 'fix the model' direction.
    """
    t = str(signal).lower()
    for tokens, guidance in GUIDANCE_HEURISTICS:
        if any(tok in t for tok in tokens):
            return guidance
    return {
        'meaning': (
            'Informational message — not a recognized code.' if is_info else
            'Unrecognized signal with no exact Learn page. Treat it as a hint that the '
            'semantic model needs tuning for this kind of question.'
        ),
        'fix': (
            'Open the overview link to categorize the message, then remediate the source model: '
            'simplify the AI schema, add measures, add AI instructions/synonyms, confirm indexing, '
            'and pin Verified Answers for common asks.'
        ),
        'link': OVERVIEW,
    }


print(f'Knowledge base covers {len(KB)} named signals + empty-result heuristic.')
print(f'Info-only signals: {sorted(INFO_SIGNALS)}')
print(f'Heuristic categories for unknown codes: {len(GUIDANCE_HEURISTICS)}')
print(f'General reference (errors/warnings/info): {OVERVIEW}')


## 6. Model AI settings (Prep data for AI)

`copilotModelSettings` is the model's AI configuration: the **AI data schema** (each field's `Visibility` — `1` visible to Copilot, `0` hidden), **AI instructions** (`CustomInstructions`), **example prompts**, and whether **indexing** is on.

In [ ]:
settings = diag.get('copilotModelSettings', {}) or {}
entities = settings.get('Entities', []) or []

tables_visible, tables_hidden = set(), set()
cols_visible, cols_hidden = [], []
terms_defined = []

for e in entities:
    b = e.get('Binding', {})
    ent = b.get('EntityName')
    prop = b.get('PropertyName')
    vis = e.get('Visibility')
    terms = e.get('Terms') or []
    if terms:
        terms_defined.append((f'{ent}[{prop}]' if prop else ent, terms))
    if prop is None:
        (tables_visible if vis == 1 else tables_hidden).add(ent)
    else:
        (cols_visible if vis == 1 else cols_hidden).append(f'{ent}[{prop}]')

print(f'Entities in AI schema     : {len(entities)}')
print(f'Tables visible to Copilot : {len(tables_visible)}')
print(f'Tables hidden from Copilot: {len(tables_hidden)}')
print(f'Columns/measures visible  : {len(cols_visible)}')
print(f'Columns/measures hidden   : {len(cols_hidden)}')
print(f'Fields with synonyms/Terms: {len(terms_defined)}')
print()
print('Indexing enabled          :', deep_get(settings, 'Settings', 'IndexingEnabled'))
print()
print('Visible tables:', sorted(tables_visible) or '—')
print()
print('Example prompts:')
for p in deep_get(settings, 'ExamplePrompts', 'Prompts', default=[]):
    print(f'  - {p}')
print()
print('AI instructions (CustomInstructions):')
ci = settings.get('CustomInstructions') or ''
print('  ' + (ci.strip().replace('\n', '\n  ') if ci.strip() else '(none set)'))

## 7. Automated troubleshooting report

Combines everything above into a prioritized list of findings and recommended model adjustments.

In [ ]:
findings = []

def add_finding(severity, signal, where, kb_entry):
    findings.append({
        'severity': severity,
        'signal': signal,
        'where': where,
        'meaning': kb_entry['meaning'],
        'fix': kb_entry['fix'],
        'link': kb_entry['link'],
    })

for q in questions:
    label = q['utterance'] or q['id']
    for e in q['errors']:
        # Prefer explicit KB, else infer the remediation category from keywords.
        kb = KB.get(e) or infer_guidance(e)
        add_finding('ERROR', f'Error: {e}', label, kb)
    for w in q['warnings']:
        is_info = w in INFO_SIGNALS
        # Explicit KB entry first; otherwise infer the closest remediation category by keyword
        # (exact codes rarely have a dedicated Learn page — see section 5).
        kb = KB.get(w) or infer_guidance(w, is_info)
        add_finding('INFO' if is_info else 'WARNING', w, label, kb)
    if q['fallback_reason'] and q['fallback_reason'] in KB:
        add_finding('INFO', f"NL->DAX fallback ({q['fallback_reason']})", label, KB[q['fallback_reason']])
    for ck in q['context_clarifications']:
        if ck in KB:
            add_finding('INFO', f'Clarification ({ck})', label, KB[ck])
    if q['empty_result'] or (q['answer'] and 'no ' in (q['answer'] or '').lower() and 'sold' in (q['answer'] or '').lower()):
        add_finding('CHECK', 'Empty/zero result', label, EMPTY_RESULT_GUIDANCE)

# Failed conversation turns (tool status other than Succeeded)
for t in turns:
    status = t.get('status')
    if status and status != 'Succeeded':
        add_finding('ERROR', f'Tool call {status}', t['question'], {
            'meaning': f"Copilot's tool call for this question did not succeed (status: {status}).",
            'fix': 'Inspect the turn in section 3. Retry after indexing; if it keeps failing, review the model (relationships/measures), rephrase the question, and check Power BI service health.',
            'link': FAQ,
        })

# Model-level checks
if deep_get(settings, 'Settings', 'IndexingEnabled') is False:
    add_finding('WARNING', 'Indexing disabled', 'Model settings', {
        'meaning': 'Q&A indexing is off, so Copilot cannot search instance values in the model.',
        'fix': 'Enable the Q&A setting on the semantic model so Power BI indexes text columns.',
        'link': FAQ,
    })

if len(entities) > 900:
    add_finding('CHECK', f'Large AI schema ({len(entities)} entities)', 'Model settings', KB['AgentSchemaReduced'])

if not (settings.get('CustomInstructions') or '').strip():
    add_finding('CHECK', 'No AI instructions', 'Model settings', {
        'meaning': 'No AI instructions are set. Copilot has no business-term or mapping context beyond names/descriptions.',
        'fix': 'Add AI instructions to define business terms, map user phrasing to fields/measures, and guide interpretation.',
        'link': TUT,
    })

order = {'ERROR': 0, 'WARNING': 1, 'CHECK': 2, 'INFO': 3}
findings.sort(key=lambda f: order.get(f['severity'], 4))

# ANSI colors so ERROR (red) and WARNING (yellow) stand out in notebook/terminal output.
RESET = '\033[0m'
SEV_COLOR = {
    'ERROR': '\033[1;91m',    # bold red
    'WARNING': '\033[1;93m',  # bold yellow
    'CHECK': '\033[96m',      # cyan
    'INFO': '\033[90m',       # grey
}

def colorize(severity, text):
    c = SEV_COLOR.get(severity, '')
    return f'{c}{text}{RESET}' if c else text

if not findings:
    print('\033[1;92mNo issues detected in this diagnostic file.\033[0m')
else:
    n_err = sum(1 for f in findings if f['severity'] == 'ERROR')
    n_warn = sum(1 for f in findings if f['severity'] == 'WARNING')
    n_info = sum(1 for f in findings if f['severity'] == 'INFO')
    err_txt = colorize('ERROR', f'{n_err} error')
    warn_txt = colorize('WARNING', f'{n_warn} warning')
    info_txt = colorize('INFO', f'{n_info} info')
    print(f'{len(findings)} finding(s)  ({err_txt}, {warn_txt}, {info_txt}):')
    print('Goal: understand why some prompts answer well and others do not, then remediate the')
    print('semantic model (the source) for consistent, high-quality Copilot answers.')
    print(f'Reference for all error / warning / info messages: {OVERVIEW}')
    print('Note: exact codes may have no dedicated Learn page — findings are mapped to the')
    print('closest remediation category by keyword (e.g. any "schema" signal -> simplify the model).\n')
    for i, f in enumerate(findings, 1):
        tag = colorize(f['severity'], f"[{f['severity']}]")
        print(f"{tag} {i}. {f['signal']}  —  ({f['where']})")
        print(f"    What it means : {f['meaning']}")
        print(f"    What to adjust: {f['fix']}")
        print(f"    Reference     : {f['link']}")
        print()


## 8. What to adjust — checklist

Apply *Prep data for AI* features in this recommended order ([FAQ](https://learn.microsoft.com/en-us/power-bi/create-reports/copilot-prepare-data-ai-faq)):

1. **Simplify the AI data schema** — hide tables/columns/measures users never query; remove duplicate or similar field names; keep within limits (~1,000 entities, 5M instance values). Prevents `AgentSchemaReduced` and wrong-field selection.
2. **Create verified answers** — pin trusted visuals to common/nuanced questions via trigger phrases. Best fix for frequent asks and for `QueryAggregateNotSupported`.
3. **Add AI instructions** — define business terms, seasons, synonyms, and term→field mappings. Best fix for `emptyInterpretation` fallbacks and terminology mismatches.
4. **Add descriptions** — table/column/measure descriptions (used by search and DAX queries). Improves discovery.
5. **Confirm indexing / Q&A enabled** — required for instance-value search; resolves `DataIndexNotReady`. Allow time for the first index (~15 min) and reindex after refresh/republish.
6. **Mark 'Approved for Copilot'** — after validating, signal the model is AI-ready.

**Also from the tutorial** ([link](https://learn.microsoft.com/en-us/power-bi/create-reports/tutorial-copilot-power-bi-prepare-model)):
- Use human-readable, unique names; follow a **star schema**; remove unused objects and ambiguous relationships.
- Add clear model, table, and column **descriptions**; add synonyms (Terms) for user vocabulary.
- **Iteratively test** with Copilot after each change and validate the answers.

> Note: For empty/zero results (e.g. "no bikes sold in 2007" while later years return values), first confirm the data actually exists for that filter; if it does, the field/term mapping or relationships likely need correcting via AI instructions and model fixes.

## 9. Export a shareable Markdown report

Assemble everything above (session metadata, model AI settings, per-query analysis, the signal summary, and the prioritized findings) into a single **Markdown report** you can hand to model owners. The file is written next to the diagnostic and also rendered inline below.

> Run sections **2–7 first** (they populate `meta`, `turns`, `questions`, `settings`, `entities`, and `findings`). Set `REPORT_MODEL_NAME` / `REPORT_WORKSPACE` to enrich the header — the diagnostic file doesn't always carry the model or workspace name.


In [ ]:
# --- 9. Build a shareable Markdown diagnostics report ---------------------------------
# Optional overrides — the diagnostic file does not always carry the model / workspace name.
REPORT_MODEL_NAME = None      # e.g. 'sales_model'
REPORT_WORKSPACE = None       # e.g. 'My Workspace'
REPORT_OUTPUT_PATH = None     # default: next to the diagnostic file

model_name = REPORT_MODEL_NAME or diag.get('modelName') or (settings.get('Name') if isinstance(settings, dict) else None) or 'semantic_model'
workspace = REPORT_WORKSPACE or diag.get('workspaceName')

# Severity -> traffic-light emoji.
SEV_EMOJI = {'ERROR': '🔴', 'WARNING': '🟡', 'CHECK': '🟠', 'INFO': '🟢'}

n_err = sum(1 for f in findings if f['severity'] == 'ERROR')
n_warn = sum(1 for f in findings if f['severity'] == 'WARNING')
n_check = sum(1 for f in findings if f['severity'] == 'CHECK')
n_info = sum(1 for f in findings if f['severity'] == 'INFO')

if n_err:
    health = '🔴 **Needs Attention**'
elif n_warn or n_check:
    health = '🟡 **Needs Optimization**'
else:
    health = '🟢 **Healthy**'

lines = []
A = lines.append

# --- Header ---------------------------------------------------------------------------
A('# Power BI Copilot — Diagnostics Report')
A('')
A(f'**Model**: `{model_name}`  ')
if workspace:
    A(f'**Workspace**: `{workspace}`  ')
A(f'**Session ID**: `{diag.get("sessionId")}`  ')
A(f'**Copilot Session ID**: `{diag.get("copilotSessionId")}`  ')
A(f'**Service version**: `{diag.get("serviceVersion")}`  ')
A(f'**Diagnostic timestamp**: `{diag.get("timestamp")}`  ')
A(f'**Report generated**: {datetime.now().strftime("%Y-%m-%d %H:%M")}  ')
A(f'**Source file**: `{os.path.basename(DIAG_PATH)}`  ')
A('')
A('---')
A('')

# --- Executive summary ----------------------------------------------------------------
A('## Executive summary')
A('')
A(f'**Overall health**: {health}')
A('')
A(
    'This report parses the Power BI Copilot diagnostic and maps each signal to a recommended '
    'semantic-model adjustment, so you can see **why some prompts answer well while others do not** '
    'and fix the source model for consistent, high-quality Copilot answers.'
)
A('')
A(
    f'It found **{n_err} error(s)**, **{n_warn} warning(s)**, **{n_check} check(s)** and '
    f'**{n_info} info** message(s) across **{len(questions)} question(s)** / '
    f'**{len(turns)} conversation turn(s)**.'
)
A('')
A(f'Reference for all Copilot error / warning / info messages: [{OVERVIEW}]({OVERVIEW})')
A('')
A(
    '> Note: exact signal codes may have no dedicated Learn page — unknown codes are mapped '
    'to the closest remediation category by keyword (e.g. any *schema* signal → simplify the model).'
)
A('')
A('---')
A('')

# --- Model AI settings ----------------------------------------------------------------
A('## Model AI settings (Prep data for AI)')
A('')
A('| Attribute | Value |')
A('|-----------|-------|')
A(f'| Entities in AI schema | {len(entities)} |')
A(f'| Tables visible to Copilot | {len(tables_visible)} |')
A(f'| Tables hidden from Copilot | {len(tables_hidden)} |')
A(f'| Columns/measures visible | {len(cols_visible)} |')
A(f'| Columns/measures hidden | {len(cols_hidden)} |')
A(f'| Fields with synonyms/Terms | {len(terms_defined)} |')
A(f'| Indexing enabled | {deep_get(settings, "Settings", "IndexingEnabled")} |')
ci = (settings.get('CustomInstructions') or '').strip()
A(f'| AI instructions set | {"Yes" if ci else "No"} |')
A('')
if tables_visible:
    A('**Visible tables**: ' + ', '.join(f'`{t}`' for t in sorted(tables_visible)))
    A('')
_prompts = deep_get(settings, 'ExamplePrompts', 'Prompts', default=[])
if _prompts:
    A('**Example prompts**:')
    A('')
    for p in _prompts:
        A(f'- {p}')
    A('')
if ci:
    A('**AI instructions (CustomInstructions)**:')
    A('')
    A('> ' + ci.replace('\n', '\n> '))
    A('')
A('---')
A('')

# --- Query-by-query analysis ----------------------------------------------------------
A('## Query-by-query analysis')
A('')
status_by_q = {t['question']: t.get('status') for t in turns}
if not questions:
    A('_No data questions found in this diagnostic._')
    A('')
for i, q in enumerate(questions, 1):
    A(f'### Query {i}: "{q["utterance"]}"')
    A('')
    A('| Attribute | Value |')
    A('|-----------|-------|')
    status = status_by_q.get(q['utterance'])
    if status:
        A(f'| Status | {"✅ Succeeded" if status == "Succeeded" else f"❌ {status}"} |')
    A(f'| Warnings | {", ".join(f"`{w}`" for w in q["warnings"]) or "—"} |')
    A(f'| Errors | {", ".join(f"`{e}`" for e in q["errors"]) or "—"} |')
    if q['fallback_reason']:
        A(f'| NL→DAX fallback | `{q["fallback_reason"]}` |')
    if q['empty_result']:
        A(f'| Empty result | {q["empty_result"]} |')
    if q['context_clarifications']:
        A(f'| Clarification | {", ".join(str(c) for c in q["context_clarifications"])} |')
    if q['answer']:
        A(f'| Answer | {str(q["answer"]).replace("|", chr(92) + "|").replace(chr(10), " ")} |')
    A('')
    if q['dax']:
        A('```dax')
        for _l in q['dax'].splitlines():
            A(_l)
        A('```')
        A('')
A('---')
A('')

# --- Signal summary -------------------------------------------------------------------
A('## Signal summary')
A('')
agg = {}
for f in findings:
    entry = agg.setdefault(f['signal'], {'severity': f['severity'], 'count': 0})
    entry['count'] += 1
if agg:
    A('| Signal | Severity | Occurrences |')
    A('|--------|----------|-------------|')
    for sig, info in sorted(agg.items(), key=lambda kv: order.get(kv[1]['severity'], 4)):
        A(f'| {sig} | {SEV_EMOJI.get(info["severity"], "")} {info["severity"]} | {info["count"]} |')
else:
    A('_No signals detected._')
A('')
A('---')
A('')

# --- Findings & recommended adjustments ----------------------------------------------
A('## Findings & recommended model adjustments')
A('')
if not findings:
    A('No issues detected in this diagnostic file. 🟢')
    A('')
else:
    for i, f in enumerate(findings, 1):
        A(f'### {SEV_EMOJI.get(f["severity"], "")} {i}. {f["signal"]} — ({f["where"]})')
        A('')
        A(f'- **What it means**: {f["meaning"]}')
        A(f'- **What to adjust**: {f["fix"]}')
        A(f'- **Reference**: {f["link"]}')
        A('')
A('---')
A('')

# --- What to adjust — checklist -------------------------------------------------------
A('## What to adjust — checklist')
A('')
A('Apply *Prep data for AI* features in this order:')
A('')
A('1. **Simplify the AI data schema** — hide tables/columns/measures users never query; remove duplicate or similar field names; keep within limits (~1,000 entities, 5M instance values).')
A('2. **Create verified answers** — pin trusted visuals to common/nuanced questions via trigger phrases.')
A('3. **Add AI instructions** — define business terms, synonyms, and term→field mappings.')
A('4. **Add descriptions** — table/column/measure descriptions used by search and DAX generation.')
A('5. **Confirm indexing / Q&A enabled** — required for instance-value search; allow ~15 min for the first index and reindex after refresh/republish.')
A("6. **Mark 'Approved for Copilot'** — after validating the answers.")
A('')
A('---')
A('')

# --- References -----------------------------------------------------------------------
A('## References')
A('')
A(f'- Copilot for Power BI overview (errors/warnings/info): {OVERVIEW}')
A(f'- FAQ: Preparing data for AI: {FAQ}')
A(f'- Tutorial: Prepare semantic model for AI: {TUT}')
A('')
A('*Report generated by the Copilot Diagnostics Troubleshooter notebook. No changes were made to the semantic model.*')

# --- Write + render -------------------------------------------------------------------
report_md = '\n'.join(lines)
_safe = ''.join(c if (c.isalnum() or c in '-_') else '_' for c in str(model_name)) or 'model'
out_path = REPORT_OUTPUT_PATH or os.path.join(os.path.dirname(DIAG_PATH), f'Copilot_Diagnostics_Report_{_safe}.md')
with open(out_path, 'w', encoding='utf-8') as fh:
    fh.write(report_md)

print(f'Markdown report written to: {out_path}')
try:
    from IPython.display import Markdown, display
    display(Markdown(report_md))
except Exception:
    print(report_md)
